In [ ]:
import time

class Z_Axis_PID:
    def __init__(self, Kp, Ki, Kd, base_throttle):
        self.Kp = Kp
        self.Ki = Ki
        self.Kd = Kd
        self.base_throttle = base_throttle # Dronun askıda kaldığı temel PWM (Örn: 1380)
        
        self.prev_error = 0
        self.integral = 0
        self.last_time = time.time()
        
    def compute(self, target_velocity, current_velocity):
        current_time = time.time()
        dt = current_time - self.last_time
        if dt <= 0.0: dt = 0.01 # Sıfıra bölünme hatasını engelle
        
        # 1. Hatayı Hesapla (Ne kadar hızlı gitmek istiyoruz eksi ne kadar hızlı gidiyoruz)
        error = target_velocity - current_velocity
        
        # 2. P, I, D bileşenleri
        P_out = self.Kp * error
        
        self.integral += error * dt
        # İntegral yığılmasını (Windup) önlemek için sınır koy
        self.integral = max(min(self.integral, 100), -100)
        I_out = self.Ki * self.integral
        
        derivative = (error - self.prev_error) / dt
        D_out = self.Kd * derivative
        
        # 3. Toplam Çıktıyı Hesapla
        output = P_out + I_out + D_out
        
        # Gelecek döngü için değerleri sakla
        self.prev_error = error
        self.last_time = current_time
        
        # Base throttle üzerine PID düzeltmesini ekle
        final_throttle = int(self.base_throttle + output)
        
        # Güvenlik sınırları (Motorları kapatma veya roket gibi fırlatma)
        return max(1100, min(final_throttle, 1700))

# --- GÖREV SENARYOSU ---
# PID katsayıları (Bunları uçuş testlerinde deneme yanılma ile bulmanız gerekir)
pid = Z_Axis_PID(Kp=1.5, Ki=0.1, Kd=0.5, base_throttle=1380)

# Pilot kumandadan ARM edip sadece motorları rölantide bırakır (Throttle=1000).
# Otonom görev başlar:

try:
    print("Otonom Yükseliş Başlıyor...")
    hedef_hiz_yukselis = 20.0 # cm/s (Yukarı)
    
    # 3 saniye boyunca yüksel
    t_end = time.time() + 3
    while time.time() < t_end:
        # 1. Sensörden mevcut dikey hızı oku (Bunu MSP üzerinden siz çekeceksiniz)
        mevcut_hiz = read_vertical_velocity_from_imu() 
        
        # 2. PID ile throttle hesapla
        yeni_throttle = pid.compute(hedef_hiz_yukselis, mevcut_hiz)
        
        # 3. Uçuş kontrolcüsüne gönder
        send_rc(throttle=yeni_throttle, aux1=2000) # AUX1=2000 (Kumanda ARM edilmiş kalmalı)
        time.sleep(0.05)

    print("Yükseklik Sabitleniyor (Hover)...")
    hedef_hiz_sabit = 0.0 # Hızı sıfırla ki olduğu yerde kalsın
    
    # 5 saniye boyunca havada kal
    t_end = time.time() + 5
    while time.time() < t_end:
        mevcut_hiz = read_vertical_velocity_from_imu() 
        yeni_throttle = pid.compute(hedef_hiz_sabit, mevcut_hiz)
        send_rc(throttle=yeni_throttle, aux1=2000)
        time.sleep(0.05)

    print("Otonom Alçalma Başlıyor...")
    hedef_hiz_alcalis = -15.0 # cm/s (Aşağı)
    
    # 3 saniye boyunca alçal
    t_end = time.time() + 3
    while time.time() < t_end:
        mevcut_hiz = read_vertical_velocity_from_imu() 
        yeni_throttle = pid.compute(hedef_hiz_alcalis, mevcut_hiz)
        send_rc(throttle=yeni_throttle, aux1=2000)
        time.sleep(0.05)

    print("Motorlar susturuluyor...")
    # Kumanda devreye girer veya gaz kesilir.
    
except KeyboardInterrupt:
    print("Acil Müdahale!")